# CFTC 纳指期货持仓官方源身份预检

只读取2011与2026两个年度TFF Futures Only压缩包，验证合约、杠杆资金字段与报告日期；发布时钟未合格，不构造交易信号。

In [1]:
import csv,io,json,hashlib,subprocess,time,tarfile,tempfile,shlex,zipfile,re
from pathlib import Path
from datetime import datetime,timezone,date
from src.data.nas_market_data import _settings,nas_api_call
root=Path.cwd();card=root/'research/stage1_cftc_nq_tff_source_probe_20261023';plan=json.loads((card/'plan.json').read_text())
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and not (card/'attempt_started.json').exists()
raw=root/'data/snapshots/stage1_cftc_nq_tff_source_probe_20261023';raw.mkdir(parents=True,exist_ok=True);assert not any(raw.iterdir())
state={'status':'started','GET_attempts':0,'market_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'attempt_started.json').write_text(json.dumps(state,indent=2)+'\n')
receipts=[];checks=[];failure=None
for i,spec in enumerate(plan['exact_requests']):
 state['GET_attempts']+=1;(card/'attempt_started.json').write_text(json.dumps(state,indent=2)+'\n');r=plan['request'];name=f"fut_fin_txt_{spec['year']}.zip";body=raw/name;headers=raw/(name+'.headers');started=datetime.now(timezone.utc).isoformat()
 cmd=['curl','--http1.1','--noproxy','*','--connect-timeout',str(r['connect_timeout_seconds']),'--max-time',str(r['total_timeout_seconds_each']),'--max-filesize',str(r['max_response_bytes_each']),'--fail-with-body','-sS','-L','-D',str(headers),'-o',str(body),'-A',r['user_agent'],'-w','%{http_code} %{url_effective}',spec['url']]
 try:
  run=subprocess.run(cmd,capture_output=True,text=True,timeout=110);parts=run.stdout.rsplit(' ',1);status=int(parts[0]) if len(parts)==2 and parts[0].isdigit() else None
  receipt={'year':spec['year'],'url':spec['url'],'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'curl_exit_code':run.returncode,'http_status':status,'final_url':parts[-1] if parts else None,'stderr':run.stderr[:350]}
 except Exception as exc:receipt={'year':spec['year'],'url':spec['url'],'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'transport_exception':type(exc).__name__+':'+str(exc)[:180]}
 receipt.update(raw_sha256=sha(body) if body.exists() else None,raw_bytes=body.stat().st_size if body.exists() else 0,headers_sha256=sha(headers) if headers.exists() else None,headers_bytes=headers.stat().st_size if headers.exists() else 0);receipts.append(receipt)
 if receipt.get('curl_exit_code')!=0 or receipt.get('http_status')!=200 or not body.exists() or body.stat().st_size>r['max_response_bytes_each']:failure='transport_or_size_failure';break
 try:
  with zipfile.ZipFile(body) as z:
   members=[m for m in z.namelist() if m.lower().endswith(('.txt','.csv')) and not m.startswith('__MACOSX/')];assert len(members)==1
   payload=z.read(members[0])
  try:decoded=payload.decode('utf-8-sig')
  except UnicodeDecodeError:decoded=payload.decode('cp1252')
  table=csv.DictReader(io.StringIO(decoded));columns=table.fieldnames;rows=list(table);assert columns and len(rows)>200
  def key(fragment):
   matches=[x for x in columns if fragment in re.sub(r'[^a-z0-9]','',x.lower())];assert len(matches)==1,(fragment,matches);return matches[0]
  market=key('marketandexchangenames');report=key('reportdateasmmddyyyy');code=key('cftccontractmarketcode');oi=key('openinterestall');long=key('levmoneypositionslongall');short=key('levmoneypositionsshortall')
  nq=[row for row in rows if re.search(r'NASDAQ\s*MINI|E[- ]MINI\s*NASDAQ',row[market],re.I) and 'CHICAGO MERCANTILE' in row[market].upper()];assert len(nq)>=20
  identity=sorted({(row[market].strip(),row[code].strip()) for row in nq});assert len({x[1] for x in identity})==1
  dates=[datetime.strptime(row[report].strip(),'%m/%d/%Y').date() for row in nq];assert len(dates)==len(set(dates)) and min(dates).year==spec['year'] and max(dates).year==spec['year']
  numeric=lambda x:float(x.replace(',','').strip())
  assert all(numeric(row[oi])>0 and numeric(row[long])>=0 and numeric(row[short])>=0 for row in nq)
  checks.append({'year':spec['year'],'status':'official_TFF_NQ_contract_identity_passed','raw_sha256':receipt['raw_sha256'],'zip_member':members[0],'zip_member_bytes':len(payload),'total_report_rows':len(rows),'NQ_rows':len(nq),'NQ_first_report_date':str(min(dates)),'NQ_last_report_date':str(max(dates)),'NQ_contract_identity':identity,'Tuesday_report_date_share':sum(d.weekday()==1 for d in dates)/len(dates),'required_column_names':{'market':market,'report_date':report,'contract_code':code,'open_interest':oi,'leveraged_money_long':long,'leveraged_money_short':short},'report_date_is_not_release_date':True})
 except Exception as exc:failure='source_identity_or_coverage_failure';checks.append({'year':spec['year'],'status':failure,'error':type(exc).__name__+':'+str(exc)[:240],'raw_sha256':receipt['raw_sha256']});break
 if i+1<len(plan['exact_requests']):time.sleep(r['seconds_between_requests'])
(card/'capture_receipts.json').write_text(json.dumps(receipts,indent=2)+'\n')
manifest={'plan_sha256':sha(card/'plan.json'),'files':[{'name':p.name,'sha256':sha(p),'bytes':p.stat().st_size} for p in sorted(raw.iterdir()) if p.is_file()],'capture_receipts_sha256':sha(card/'capture_receipts.json')};(card/'source_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
settings=_settings(root)
with tempfile.TemporaryDirectory(prefix='cftc-tff-source-') as td:
 archive=Path(td)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as t:
  for x in manifest['files']:t.add(raw/x['name'],arcname='raw/'+x['name'])
  t.add(card/'source_manifest.json',arcname='source_manifest.json');t.add(card/'capture_receipts.json',arcname='capture_receipts.json')
 digest=sha(archive);evidence=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact'];assert evidence['sha256']==digest and evidence['size_bytes']==archive.stat().st_size
 remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
 with tempfile.TemporaryDirectory(prefix='cftc-tff-restore-') as rd:
  restored=Path(rd)/'restore.tar.gz'
  with restored.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(remote)],stdout=out,stderr=subprocess.PIPE,timeout=120)
  assert run.returncode==0 and sha(restored)==digest
  with tarfile.open(restored,'r:gz') as t:
   for x in manifest['files']:
    member=t.extractfile('raw/'+x['name']);assert member and hashlib.sha256(member.read()).hexdigest()==x['sha256']
result={'status':'official_CFTC_TFF_NQ_2011_2026_source_identity_passed_release_clock_pending' if failure is None and len(checks)==2 else 'bounded_CFTC_TFF_NQ_source_partial_or_failed','failure':failure,'GET_attempts':state['GET_attempts'],'checks':checks,'NAS_archive_sha256':digest,'NAS_remote_and_scoped_restore_verified':True,'market_outcome_reads':0,'model_fits':0,'native_accounts':0,'historical_release_clock_qualified':False};(card/'source_result.json').write_text(json.dumps(result,indent=2)+'\n');print({'status':result['status'],'failure':failure,'GETs':state['GET_attempts'],'checks':checks,'NAS_sha256':digest})


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'status': 'bounded_CFTC_TFF_NQ_source_partial_or_failed', 'failure': 'source_identity_or_coverage_failure', 'GETs': 1, 'checks': [{'year': 2011, 'status': 'source_identity_or_coverage_failure', 'error': "AssertionError:('cftccontractmarketcode', ['CFTC_Contract_Market_Code', 'CFTC_Contract_Market_Code_Quotes'])", 'raw_sha256': '583a93474847f1070b213e2b12cfa73c48d9ba05f24f7d001a338de125468b57'}], 'NAS_sha256': 'e81deef98e984c4c68292e0cb13a17f4788369f0b5e1c3ab7db62459e7624509'}


## 中文结论

首次2011 GET与 NAS 备份成功，解析器因合约代码列歧义失败；后续修复结果见 `schema_alias_result.json`。

完整科学决定见 `conclusion.md`。